# 3. Custom Tools for a Customer Outreach Campaign

**Learning objectives**
- Understand why agents need tools.
- Use a web-search tool when a Serper API key is available.
- Create a custom CrewAI tool with a validated input schema.
- Use a deterministic sentiment-analysis demonstration.
- Generate email drafts while keeping a human in control of sending.

# Setup: CrewAI + Groq (Colab)

**Before running:** add a Groq API key to Google Colab → Secrets with the name `GROQ_API_KEY` and enable notebook access to it. A Serper key (`SERPER_API_KEY`) is optional and is only needed for live web search in the relevant exercise.

These notebooks use the current CrewAI Python API style with Groq's `openai/gpt-oss-120b` model (called through LiteLLM). The exact package versions are intentionally not pinned; after a successful lab setup, freeze the working environment for repeatability.

In [ ]:
%pip -q install -U "crewai[litellm]" crewai-tools groq

In [ ]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Add GROQ_API_KEY in Colab Secrets and enable notebook access.")

llm = LLM(
    model="groq/openai/gpt-oss-120b",
    api_key=GROQ_API_KEY,
    temperature=0.2
)
print("LLM configured (Groq).")

LLM configured (Groq).


## Step 1 — Configure an optional web-search tool

Add `SERPER_API_KEY` to Colab Secrets to enable live search. If it is not configured, this notebook still runs using the local demo tool and supplied fictional company brief.

In [ ]:
from crewai_tools import SerperDevTool

try:
    SERPER_API_KEY = userdata.get("SERPER_API_KEY")
except Exception:
    SERPER_API_KEY = None

search_tool = None

if SERPER_API_KEY:
    os.environ["SERPER_API_KEY"] = SERPER_API_KEY
    search_tool = SerperDevTool(n_results=3)
    print("Live web search enabled.")
else:
    print("SERPER_API_KEY not found: live search disabled.")

Live web search enabled.


## Step 2 — Build a custom sentiment tool

This is a transparent **rule-based teaching example**, not a trained sentiment model. It counts simple positive and negative words. Discuss its limitations: negation, sarcasm, context and domain-specific language.

In [ ]:
import re
from pydantic import BaseModel, Field
from crewai.tools import BaseTool

class SentimentInput(BaseModel):
    text: str = Field(..., description="Text whose sentiment should be estimated.")

class SimpleSentimentTool(BaseTool):
    name: str = "Simple Rule-Based Sentiment"
    description: str = (
        "Estimate text sentiment using a small positive/negative word list. "
        "This is a simple demonstration, not a production sentiment model."
    )
    args_schema: type[BaseModel] = SentimentInput

    def _run(self, text: str) -> str:
        positive_words = {
            "good", "great", "excellent", "helpful", "success",
            "interested", "pleased", "improved", "benefit", "love"
        }
        negative_words = {
            "bad", "poor", "issue", "problem", "delay",
            "disappointed", "difficult", "failed", "hate", "complaint"
        }

        tokens = set(re.findall(r"[a-z']+", text.lower()))
        positive = len(tokens & positive_words)
        negative = len(tokens & negative_words)

        if positive > negative:
            label = "positive"
        elif negative > positive:
            label = "negative"
        else:
            label = "neutral"

        return (
            f"Estimated sentiment: {label}. "
            f"Positive keyword matches: {positive}; negative keyword matches: {negative}. "
            "This is a rule-based estimate and may miss context or sarcasm."
        )

sentiment_tool = SimpleSentimentTool()
print(sentiment_tool.run(text="The support team was helpful and the launch was great."))

Estimated sentiment: positive. Positive keyword matches: 2; negative keyword matches: 0. This is a rule-based estimate and may miss context or sarcasm.


## Step 3 — Define the agents

The first agent profiles the organisation. The second drafts a personalized message based on the profile. Do not assume that a named individual is the decision-maker unless the source confirms it.

In [ ]:
RESEARCH_MODEL = "groq/qwen/qwen3.8-27b"
researcher_llm = LLM(model=RESEARCH_MODEL, api_key=GROQ_API_KEY, temperature=0.2)

tools_for_research = [search_tool] if search_tool else []

lead_researcher = Agent(
    role="Organisation Research Analyst",
    goal="Prepare a concise, evidence-aware profile of {lead_name}",
    backstory=(
        "You research organisations using available tools. Separate verified "
        "facts from assumptions, include source links when web search is used, "
        "and report when information cannot be verified. Use at most 2 searches, "
        "then write your answer. Only use tools you were explicitly given; "
        "never try to open files or browse pages in any other way."
    ),
    tools=tools_for_research,
    llm=researcher_llm,
    verbose=True,
    allow_delegation=False,
    max_iter=8
)

outreach_writer = Agent(
    role="Customer Outreach Writer",
    goal="Create a relevant, respectful and personalized outreach email draft",
    backstory=(
        "You write professional outreach. Use only facts present in the "
        "research brief or supplied inputs. Do not invent personal details, "
        "relationships, product claims or recent milestones."
    ),
    tools=[sentiment_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

## Step 4 — Define the tasks

In [ ]:
profile_task = Task(
    description=(
        "Prepare a profile of {lead_name}, operating in {industry}. "
        "Use web search if available. If live search is unavailable, rely only "
        "on the supplied brief and clearly state that independent verification "
        "has not been performed. Do not infer who the decision-maker is."
    ),
    expected_output=(
        "A short profile with verified facts, source URLs if available, "
        "possible needs and a separate list of unknowns."
    ),
    agent=lead_researcher
)

outreach_task = Task(
    description=(
        "Using the profile of {lead_name}, draft an outreach email to "
        "{contact_name}, whose supplied role is {contact_role}. "
        "The supplied context is: {context}. "
        "Do not add unsupported facts. Use the sentiment tool to review the "
        "tone of the draft. Return a subject line and email body only."
    ),
    expected_output="One concise, personalized email draft with a subject line and body.",
    agent=outreach_writer,
    tools=[sentiment_tool],
    markdown=True
)

## Step 5 — Assemble and run the Crew

`human_input=True` asks for human review at the task stage in supported execution environments. In this exercise, treat the result as a draft: do not connect an email-sending tool.

In [ ]:
outreach_crew = Crew(
    agents=[lead_researcher, outreach_writer],
    tasks=[profile_task, outreach_task],
    process=Process.sequential,
    verbose=True
)

# Fictional example data for classroom use.
inputs = {
    "lead_name": "Northstar Learning Pvt Ltd",
    "industry": "Online education",
    "contact_name": "Aarav Mehta",
    "contact_role": "Partnerships Manager",
    "context": (
        "The organisation is exploring ways to improve learner engagement. "
        "This is a fictional scenario supplied for the exercise, not a verified milestone."
    )
}

#outreach_result = outreach_crew.kickoff(inputs=inputs)
outreach_result = await outreach_crew.kickoff_async(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 95978556-6a75-4f1c-997e-a99e22320491                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Prepare a profile of Northstar Learning Pvt Ltd, operating in Online education. Use web search if        │
│  available. If live search is unavailable, rely only on the supplied brief and clearly state that independent   │
│  verification has not been performed. Do not infer who the decision-maker is.                                   │
│  ID: 34e81e1b-3dc0-45f1-b470-be4b6b5e1b9a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Organisation Research Analyst                                                                           │
│                                                                                                                 │
│  Task: Prepare a profile of Northstar Learning Pvt Ltd, operating in Online education. Use web search if        │
│  available. If live search is unavailable, rely only on the supplied brief and clearly state that independent   │
│  verification has not been performed. Do not infer who the decision-maker is.                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': '"Northstar Learning" Pvt Ltd company profile'}                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': '"Northstar Learning Pvt Ltd" online education India'}                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Output: {'searchParameters': {'q': '"Northstar Learning Pvt Ltd" online education India', 'type': 'search',    │
│  'num': 3, 'engine': 'google'}, 'organic': [], 'credits': 1}                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (10 outputs left out)

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Outreach Writer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Subject: Enhancing Learner Engagement at Northstar Learning                                                  │
│                                                                                                                 │
│  Hi Aarav,                                                                                                      │
│                                                                                                                 │
│  I hope you’re doing well. I’m reaching out because I understand Northstar Learning is exploring ways to boost  │
│  learner engagement. At **[Your Company]**, we specialize in designing interactive learning experiences that    │
│  keep students motivated and active throughout their courses.                                                   │
│                                                                                                                 │
│  Our approach includes:                                                                                         │
│  - **Gamified modules** that reward progress                                                                    │
│  - **Real‑time analytics** to identify engagement gaps                                                          │
│  - **Personalized content pathways** tailored to individual learner needs                                       │
│                                                                                                                 │
│  I’d love to discuss how we could support Northstar Learning’s goals and explore a partnership that aligns      │
│  with your vision. Would you be available for a brief call next week?                                           │
│                                                                                                                 │
│  Thank you for considering this, and I look forward to the possibility of collaborating.                        │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│  [Your Name]                                                                                                    │
│  [Your Title]                                                                                                   │
│  [Your Company]                                                                                                 │
│  [Contact Information]                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the profile of Northstar Learning Pvt Ltd, draft an outreach email to Aarav Mehta, whose supplied  │
│  role is Partnerships Manager. The supplied context is: The organisation is exploring ways to improve learner   │
│  engagement. This is a fictional scenario supplied for the exercise, not a verified milestone.. Do not add      │
│  unsupported facts. Use the sentiment tool to review the tone of the draft. Return a subject line and email     │
│  body only.                                                                                                     │
│  Agent: Customer Outreach Writer                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 95978556-6a75-4f1c-997e-a99e22320491                                                                       │
│  Final Output: # Subject: Enhancing Learner Engagement at Northstar Learning                                    │
│                                                                                                                 │
│  Hi Aarav,                                                                                                      │
│                                                                                                                 │
│  I hope you’re doing well. I’m reaching out because I understand Northstar Learning is exploring ways to boost  │
│  learner engagement. At **[Your Company]**, we specialize in designing interactive learning experiences that    │
│  keep students motivated and active throughout their courses.                                                   │
│                                                                                                                 │
│  Our approach includes:                                                                                         │
│  - **Gamified modules** that reward progress                                                                    │
│  - **Real‑time analytics** to identify engagement gaps                                                          │
│  - **Personalized content pathways** tailored to individual learner needs                                       │
│                                                                                                                 │
│  I’d love to discuss how we could support Northstar Learning’s goals and explore a partnership that aligns      │
│  with your vision. Would you be available for a brief call next week?                                           │
│                                                                                                                 │
│  Thank you for considering this, and I look forward to the possibility of collaborating.                        │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│  [Your Name]                                                                                                    │
│  [Your Title]                                                                                                   │
│  [Your Company]                                                                                                 │
│  [Contact Information]                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
from IPython.display import Markdown, display
display(Markdown(outreach_result.raw))

# Subject: Enhancing Learner Engagement at Northstar Learning  

Hi Aarav,

I hope you’re doing well. I’m reaching out because I understand Northstar Learning is exploring ways to boost learner engagement. At **[Your Company]**, we specialize in designing interactive learning experiences that keep students motivated and active throughout their courses.

Our approach includes:
- **Gamified modules** that reward progress  
- **Real‑time analytics** to identify engagement gaps  
- **Personalized content pathways** tailored to individual learner needs  

I’d love to discuss how we could support Northstar Learning’s goals and explore a partnership that aligns with your vision. Would you be available for a brief call next week?

Thank you for considering this, and I look forward to the possibility of collaborating.

Best regards,  
[Your Name]  
[Your Title]  
[Your Company]  
[Contact Information]

## Step 6 — Human review before use

The crew produces a draft only. Review the recipient, factual claims, tone and relevance manually. This simple cell records a classroom approval decision; it does not send an email or call an external service.

In [ ]:
approval = input("After reviewing the draft, type APPROVE to mark it ready for manual use: ").strip()
if approval == "APPROVE":
    print("Marked as reviewed. No email has been sent.")
else:
    print("Not approved. Revise the draft before use.")

After reviewing the draft, type APPROVE to mark it ready for manual use: APPROVE
Marked as reviewed. No email has been sent.


## Try it yourself
1. Test the sentiment tool with positive, negative, mixed and sarcastic sentences.
2. Add a custom tool that checks whether an email contains a greeting, clear purpose and closing.
3. Run with and without Serper search and compare the evidence quality.
4. Add a human approval step before any external communication.

**Safety rule:** Keep the workflow in draft mode. Never send bulk or personalized messages automatically from a classroom exercise.

# Exercise solutions — Try it yourself



## Exercise 1 — Test the sentiment tool with positive, negative, mixed and sarcastic sentences

The tool is rule-based, so we expect it to fail on some cases. The table compares what a human would say with what the tool says.

In [ ]:
import re, time
import pandas as pd
from IPython.display import Markdown, display

tests = [
    ("positive",  "The onboarding was excellent and the support team was very helpful."),
    ("negative",  "I am disappointed. The delay was a big problem and I hate the poor quality."),
    ("mixed",     "The product is great, but the delivery had a problem and a long delay."),
    ("sarcastic", "Oh great, another delay. Just what we needed, thanks a lot."),
    ("negation",  "The service was not good at all."),
]
human_label = {"positive": "positive", "negative": "negative", "mixed": "mixed",
               "sarcastic": "negative", "negation": "negative"}

rows = []
for kind, text in tests:
    out = sentiment_tool.run(text=text)
    label = re.search(r"Estimated sentiment: (\w+)", out).group(1)
    rows.append({
        "case": kind,
        "sentence": text,
        "human_label": human_label[kind],
        "tool_label": label,
        "correct": label == human_label[kind],
    })

pd.set_option("display.max_colwidth", 80)
display(pd.DataFrame(rows))

,case,sentence,human_label,tool_label,correct
0,positive,The onboarding was excellent and the support team was very helpful.,positive,positive,True
1,negative,I am disappointed. The delay was a big problem and I hate the poor quality.,negative,negative,True
2,mixed,"The product is great, but the delivery had a problem and a long delay.",mixed,negative,False
3,sarcastic,"Oh great, another delay. Just what we needed, thanks a lot.",negative,neutral,False
4,negation,The service was not good at all.,negative,positive,False


**What this shows:** the tool handles clearly positive and clearly negative text, but it
- cannot represent **mixed** sentiment (it just picks whichever count is larger),
- misses **sarcasm** (the positive word "great" cancels the negative word "delay", giving neutral), and
- ignores **negation** ("not good" counts as a positive word).

## Exercise 2 — Add a custom tool that checks greeting, clear purpose and closing

`EmailStructureCheckerTool` uses a validated input schema (like the sentiment tool). We test it on a good and a bad draft, then give it to a new writer agent and run the crew.

In [ ]:
from pydantic import BaseModel, Field
from crewai.tools import BaseTool

class EmailCheckInput(BaseModel):
    email_text: str = Field(..., description="The full email draft (subject line and body).")

class EmailStructureCheckerTool(BaseTool):
    name: str = "Email Structure Checker"
    description: str = (
        "Check whether an email draft contains a greeting, a clear purpose and a "
        "closing. Returns PASS or FAIL for each element and an overall result."
    )
    args_schema: type[BaseModel] = EmailCheckInput

    def _run(self, email_text: str) -> str:
        text = email_text.lower()
        lines_ = [l.strip().lower() for l in email_text.strip().splitlines() if l.strip()]
        tail = " ".join(lines_[-5:])

        greeting_words = ["hi ", "hello", "dear ", "hey ", "good morning", "good afternoon", "good evening"]
        purpose_phrases = [
            "reaching out", "i am writing", "i'm writing", "i would like to", "i'd like to",
            "i wanted to", "would love to", "i am contacting", "to discuss", "to explore",
            "to introduce", "to propose", "to schedule", "my purpose",
        ]
        closing_phrases = [
            "best regards", "kind regards", "warm regards", "regards", "sincerely",
            "best wishes", "thank you", "thanks", "cheers", "yours truly",
        ]

        has_greeting = any(g in text for g in greeting_words)
        has_purpose = any(p in text for p in purpose_phrases)
        has_closing = any(c in tail for c in closing_phrases) or any(l in ("best,", "best") for l in lines_[-5:])

        def flag(ok): return "PASS" if ok else "FAIL"
        overall = has_greeting and has_purpose and has_closing
        return (
            f"Greeting: {flag(has_greeting)}; Purpose: {flag(has_purpose)}; "
            f"Closing: {flag(has_closing)}; Overall: {flag(overall)}."
        )

email_checker_tool = EmailStructureCheckerTool()

good_email = """Subject: Exploring a partnership on learner engagement

Hello Aarav,

I am reaching out to explore how we might collaborate on improving learner engagement.
Would a short call next week suit you?

Kind regards,
Priya"""

bad_email = """Subject: Quick thing

We have a product. It is great. Buy it."""

print("GOOD:", email_checker_tool.run(email_text=good_email))
print("BAD :", email_checker_tool.run(email_text=bad_email))

GOOD: Greeting: PASS; Purpose: PASS; Closing: PASS; Overall: PASS.
BAD : Greeting: FAIL; Purpose: FAIL; Closing: FAIL; Overall: FAIL.


In [ ]:
import requests
r = requests.get("https://api.groq.com/openai/v1/models",
                 headers={"Authorization": f"Bearer {GROQ_API_KEY}"})
for m in r.json()["data"]:
    print(m["id"])

canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
whisper-large-v3
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
qwen/qwen3.8-27b
allam-2-7b
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b


In [ ]:
import time, re
import pandas as pd
from IPython.display import Markdown, display

# Fresh-quota models (each Groq model has its own daily limit)
llm = LLM(model="groq/openai/gpt-oss-20b", api_key=GROQ_API_KEY, temperature=0.2)
researcher_llm = LLM(model="groq/qwen/qwen3.8-27b", api_key=GROQ_API_KEY, temperature=0.2)


def build_outreach_crew(use_search, inputs_lead, with_checker=True):
    """Fresh researcher + writer + tasks. use_search=True gives the researcher the Serper tool."""
    researcher_tools = [search_tool] if (use_search and search_tool) else []

    researcher = Agent(
        role="Organisation Research Analyst",
        goal="Prepare a concise, evidence-aware profile of {lead_name}",
        backstory=(
            "You research organisations using only the tools you were explicitly given. "
            "Never call browser, search or any tool that is not in your list. "
            "If you have no tools, write a short profile using ONLY the supplied inputs "
            "and state clearly that independent verification has not been performed. "
            "Separate verified facts from assumptions, include source links when web "
            "search is used, and report when information cannot be verified. "
            "Use at most 2 searches, then write your answer."
        ),
        tools=researcher_tools,
        llm=researcher_llm, verbose=False, allow_delegation=False, max_iter=5,
    )

    writer_tools = [sentiment_tool] + ([email_checker_tool] if with_checker else [])
    writer = Agent(
        role="Customer Outreach Writer",
        goal="Create a relevant, respectful and personalized outreach email draft",
        backstory=(
            "You write professional outreach. Use only facts present in the "
            "research brief or supplied inputs. Do not invent personal details, "
            "relationships, product claims, services, results or recent milestones. "
            "Do not describe what the sender's company offers; use the placeholder "
            "[Your Company] only."
        ),
        tools=writer_tools,
        llm=llm, verbose=False, allow_delegation=False, max_iter=6,
    )

    p_task = Task(
        description=(
            "Prepare a profile of {lead_name}, operating in {industry}. "
            "Use web search if available. If live search is unavailable, rely only "
            "on the supplied brief and clearly state that independent verification "
            "has not been performed. Do not infer who the decision-maker is."
        ),
        expected_output=(
            "A short profile with verified facts, source URLs if available, "
            "possible needs and a separate list of unknowns."
        ),
        agent=researcher,
    )

    checker_line = (
        "Then run the Email Structure Checker on your draft and fix any FAIL before answering. "
        if with_checker else ""
    )
    o_task = Task(
        description=(
            "Using the profile of {lead_name}, draft an outreach email to "
            "{contact_name}, whose supplied role is {contact_role}. "
            "The supplied context is: {context}. "
            "Do not add unsupported facts, services or past results. "
            "Use the sentiment tool to review the tone of the draft. " + checker_line +
            "Return a subject line and email body only."
        ),
        expected_output="One concise, personalized email draft with a subject line and body.",
        agent=writer,
        context=[p_task],
        markdown=True,
    )
    return Crew(agents=[researcher, writer], tasks=[p_task, o_task],
                process=Process.sequential, verbose=False)


async def run_outreach(crew, inputs):
    start = time.time()
    result = await crew.kickoff_async(inputs=inputs)
    return result, round(time.time() - start, 1)


FICTIONAL_INPUTS = {
    "lead_name": "Northstar Learning Pvt Ltd",
    "industry": "Online education",
    "contact_name": "Aarav Mehta",
    "contact_role": "Partnerships Manager",
    "context": (
        "The organisation is exploring ways to improve learner engagement. "
        "This is a fictional scenario supplied for the exercise, not a verified milestone."
    ),
}

crew_ex2 = build_outreach_crew(use_search=False, inputs_lead=FICTIONAL_INPUTS, with_checker=True)
result_ex2, secs_ex2 = await run_outreach(crew_ex2, FICTIONAL_INPUTS)
display(Markdown(result_ex2.raw))
print(f"({secs_ex2} s)")

# Independent verification of the final draft with the new tool:
print("\nChecker on final draft ->", email_checker_tool.run(email_text=result_ex2.raw))

**Subject:** Quick chat about boosting learner engagement at Northstar Learning  

Hi Aarav,

I hope this message finds you well. I’ve been following Northstar Learning’s work in online education and understand you’re currently exploring ways to enhance learner engagement. I’d love to share some insights on how collaborative partnerships can support that goal.

Would you be open to a brief call next week to discuss potential synergies? I’m happy to accommodate your schedule.

Thank you for your time, and I look forward to connecting.

Best regards,  
[Your Name]  
[Your Company]

(3.3 s)

Checker on final draft -> Greeting: PASS; Purpose: PASS; Closing: PASS; Overall: PASS.


## Exercise 3 — Run with and without Serper search and compare evidence quality

For a meaningful comparison we use a **real, public organisation** (the fictional company would return nothing useful from a search). The email is only a draft and is never sent. If `SERPER_API_KEY` is not set, only the *without search* run is performed.

In [ ]:
COMPARE_INPUTS = {
    "lead_name": "Coursera",
    "industry": "Online education",
    "contact_name": "the Partnerships team",
    "contact_role": "supplied role: partnerships (unverified)",
    "context": (
        "Classroom exercise: we are exploring ways to support learner engagement. "
        "No recent milestones are supplied."
    ),
}

results3 = {}

crew_nosearch = build_outreach_crew(use_search=False, inputs_lead=COMPARE_INPUTS, with_checker=False)
results3["without_search"] = await run_outreach(crew_nosearch, COMPARE_INPUTS)

if search_tool:
    crew_search = build_outreach_crew(use_search=True, inputs_lead=COMPARE_INPUTS, with_checker=False)
    results3["with_search"] = await run_outreach(crew_search, COMPARE_INPUTS)
else:
    print("SERPER_API_KEY not set: skipping the WITH-search run. "
          "Add the key in Colab Secrets and re-run to complete this exercise.")

rows = []
for name, (res, secs) in results3.items():
    profile = res.tasks_output[0].raw          # output of the research task
    urls = set(re.findall(r"https?://[^\s)\]>]+", profile))
    rows.append({
        "run": name,
        "seconds": secs,
        "profile_words": len(profile.split()),
        "source_urls": len(urls),
        "says_unverified": any(k in profile.lower() for k in ["not been performed", "unverified", "cannot be verified", "not verified"]),
    })
    display(Markdown(f"### Profile — {name}\n\n{profile}"))

display(pd.DataFrame(rows))

### Profile — without_search

**Profile: Coursera**

**Status of Verification:**
Independent verification has **not** been performed. I do not have access to web search tools or external databases in this environment. The following profile is constructed **exclusively** from the supplied brief and general public knowledge inherent to the entity’s identity as a major online education platform. All specific metrics, recent financial data, or strategic shifts are marked as unverified or unknown.

---

### 1. Verified Facts (Based on General Public Knowledge & Supplied Context)
*Note: These are widely established facts about Coursera’s existence and core business model. They are not derived from a live search in this session.*

*   **Industry:** Online Education / EdTech.
*   **Core Business Model:** Coursera operates as a marketplace and platform connecting learners with courses, professional certificates, and degrees from leading universities and companies.
*   **Key Partnerships:** Historically known for partnerships with major universities (e.g., Stanford, Yale, University of Pennsylvania) and technology companies (e.g., Google, IBM, DeepLearning.AI).
*   **Offerings:**
    *   Individual courses (audit for free or paid certificate).
    *   Specializations (series of courses).
    *   Professional Certificates.
    *   Online Degrees (Master’s and Bachelor’s programs).
    *   Coursera Plus (subscription model for unlimited access to courses).
    *   Coursera for Business (enterprise solutions).
*   **Founding:** Founded in 2012 by Andrew Ng and Daphne Koller (Stanford University professors).
*   **Headquarters:** Mountain View, California, USA.
*   **Public Status:** Publicly traded company (NASDAQ: COUR).

### 2. Possible Needs (Inferred from Industry Position)
*These are strategic hypotheses based on the nature of the online education sector and Coursera’s known position. They are **not** verified current internal goals.*

*   **Enterprise Adoption:** Increasing penetration in the B2B market (Coursera for Business) to diversify revenue beyond individual subscriptions.
*   **Credential Value:** Enhancing the perceived value and employer recognition of its certificates and degrees to compete with traditional higher education and other EdTech platforms (e.g., edX, Udacity, LinkedIn Learning).
*   **AI Integration:** Leveraging AI for personalized learning paths, automated grading, and content creation to improve learner outcomes and reduce costs.
*   **Global Expansion:** Expanding access in emerging markets where digital infrastructure is improving but traditional education access is limited.
*   **Content Differentiation:** Securing exclusive or high-demand content from top-tier institutions and industry leaders to maintain competitive advantage.

### 3. Unknowns (Cannot Be Verified Without Live Search)
*The following specific details require real-time data and are **not** provided in the supplied brief:*

*   **Current Financial Metrics:** Latest quarterly revenue, profit/loss, and user growth rates.
*   **Recent Strategic Shifts:** Any new partnerships, acquisitions, or discontinuations of programs in the last 6–12 months.
*   **Current Leadership:** Specific names of current C-suite executives (CEO, CFO, etc.) if there have been recent changes.
*   **Competitive Landscape Changes:** How recent moves by competitors (e.g., LinkedIn, Udemy, edX) have impacted Coursera’s market share.
*   **Specific Product Updates:** New features launched in the past year (e.g., specific AI tools, new degree programs).
*   **Regulatory Issues:** Any ongoing legal or regulatory challenges in specific regions.

---

**Disclaimer:** This profile is a structural overview based on the entity’s known identity. It does not constitute financial advice or a current market analysis. For up-to-date information, please consult Coursera’s official investor relations page, recent press releases, or reputable financial news sources.

### Profile — with_search

# Coursera — Organisation Profile

**Sector:** Online education / EdTech (MOOC platform)

## Verified Facts (with sources)

- **What it is:** Coursera Inc. is an American online learning platform, originally known for offering massive open online courses (MOOCs). It was founded in 2012 by Stanford faculty (Andrew Ng and Daphne Koller).
  - Source: https://en.wikipedia.org/wiki/Coursera
- **Financials (FY2024):** Total revenue was **$179.2 million, up 6%** from $168.9 million the prior year (Q4/full-year 2024 results).
  - Source: https://investor.coursera.com/news/news-details/2025/Coursera-Reports-Fourth-Quarter-and-Full-Year-2024-Financial-Results/default.aspx
- **Scale:** Coursera reported **over 136 million registered learners** by 2024.
  - Source: https://thestrategicanalyst.substack.com/p/why-courseras-university-partnerships
- **Partnerships:** Partnerships with **more than 300 universities and organizations**; the 2024 annual report notes the company added 25+ new educator partners to end the year with **over 350 world-class universities**.
  - Sources: https://thestrategicanalyst.substack.com/p/why-courseras-university-partnerships ; https://www.sec.gov/Archives/edgar/data/1651562/000165156225000029/a2024annualreportdraftfina.pdf
- **Business model:** Relies on educational content partnerships; offers online degree programs and courses, with a partnership model that "amplifies content expertise" while charging one of the lower service fees for online degree programs.
  - Sources: https://www.coursera.org/partnerships/university ; https://www.sec.gov/Archives/edgar/data/1651562/000165156225000029/a2024annualreportdraftfina.pdf
- **Publicly traded:** Coursera files with the SEC (CIK 1651562), indicating it is a public company.
  - Source: https://www.sec.gov/Archives/edgar/data/1651562/000165156225000029/a2024annualreportdraftfina.pdf

## Possible Needs (inferred, not verified)
- **Content/partnership expansion:** Continued growth in educator and university partners to broaden course and degree offerings.
- **Learner monetization:** Converting its large registered-learner base (136M+) into paying subscribers (Coursera Plus) and degree students.
- **Enterprise/B2B demand:** Growing corporate training and upskilling offerings to diversify revenue.
- **Technology/AI:** Likely investment in AI-driven personalization, course creation, and learner support to stay competitive.
- **Margin improvement:** Sustaining growth while managing the cost of hosting and licensing content.

> Note: The "needs" above are reasonable inferences from the verified business model and are **not independently confirmed** by the sources.

## Unknowns / Could Not Be Verified
- **Current CEO / leadership team** — not confirmed in the sources retrieved (do not infer a specific decision-maker).
- **Exact employee count** — not stated in the retrieved results.
- **Headquarters location** — not confirmed in the retrieved snippets (commonly cited as Mountain View, CA, but not verified here).
- **Full-year 2024 net income / profitability status** — revenue was reported, but profitability was not in the snippets.
- **2025 financial performance** — beyond the FY2024 figures, no verified data retrieved.
- **Specific revenue breakdown** (subscriptions vs. degrees vs. enterprise) — not available in the retrieved results.

---
*Independent verification was performed via web search. Facts above are tied to the cited sources; items under "Possible Needs" and "Unknowns" are clearly separated from verified facts.*

,run,seconds,profile_words,source_urls,says_unverified
0,without_search,4.1,526,0,True
1,with_search,50.4,418,5,True


## Exercise 4 — Add a human approval step before any external communication

The gate (1) runs the structure checker, (2) shows the draft, (3) asks a human to type `APPROVE`, and (4) records the decision in an audit log. The send function is a **stub**: it refuses to run unless the draft was approved, and even then it only prints a message. *(CrewAI's `Task(human_input=True)` offers a similar review inside the task itself.)*

In [ ]:
approval_log = []

def human_approval_gate(draft_text, recipient):
    check = email_checker_tool.run(email_text=draft_text)
    print("Automatic structure check:", check)
    display(Markdown("### Draft for human review\n\n" + draft_text))
    decision = input("Type APPROVE to approve this draft (anything else rejects it): ").strip()
    approved = (decision == "APPROVE")
    approval_log.append({
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "recipient": recipient,
        "approved": approved,
        "structure_check": check,
    })
    return approved

def send_email_stub(draft_text, recipient, approved):
    """Stub only. Never sends anything; refuses unless a human approved the draft."""
    if not approved:
        raise PermissionError("Blocked: this draft has not been approved by a human.")
    print(f"[STUB] Would send the approved draft to {recipient}. Nothing was actually sent.")

draft = result_ex2.raw                       # draft produced in Exercise 2
recipient = "Aarav Mehta (fictional)"

# 1) Trying to send without approval must be blocked
try:
    send_email_stub(draft, recipient, approved=False)
except PermissionError as e:
    print(e)

# 2) Human review, then (maybe) the stub send
approved = human_approval_gate(draft, recipient)
try:
    send_email_stub(draft, recipient, approved)
except PermissionError as e:
    print(e)

display(pd.DataFrame(approval_log))

Blocked: this draft has not been approved by a human.
Automatic structure check: Greeting: PASS; Purpose: PASS; Closing: PASS; Overall: PASS.


### Draft for human review

**Subject:** Quick chat about boosting learner engagement at Northstar Learning  

Hi Aarav,

I hope this message finds you well. I’ve been following Northstar Learning’s work in online education and understand you’re currently exploring ways to enhance learner engagement. I’d love to share some insights on how collaborative partnerships can support that goal.

Would you be open to a brief call next week to discuss potential synergies? I’m happy to accommodate your schedule.

Thank you for your time, and I look forward to connecting.

Best regards,  
[Your Name]  
[Your Company]

Type APPROVE to approve this draft (anything else rejects it): APPROVE
[STUB] Would send the approved draft to Aarav Mehta (fictional). Nothing was actually sent.


,time,recipient,approved,structure_check
0,2026-11-25 23:02:53,Aarav Mehta (fictional),True,Greeting: PASS; Purpose: PASS; Closing: PASS; Overall: PASS.


## Exercise observations

**Ex 1 – Sentiment tool:** The tool classified 2 of 5 sentences correctly. It got the clearly positive and clearly negative sentences right. It failed on three cases:
- **Mixed:** "great… but problem and delay" was labelled negative, because the tool only compares keyword counts and cannot represent mixed sentiment.
- **Sarcasm:** "Oh great, another delay…" was labelled neutral, because the positive word "great" cancelled the negative word "delay".
- **Negation:** "not good at all" was labelled positive, because the tool sees "good" and ignores "not".

A rule-based word list is therefore only a demonstration, not a reliable sentiment model.

**Ex 2 – Email checker:** I built an `EmailStructureCheckerTool` that checks for a greeting, a clear purpose and a closing. It passed a good example email and failed a bad one. The crew's final draft also passed (Greeting PASS, Purpose PASS, Closing PASS) and contained no invented services. The earlier Step 5 draft, however, listed "gamified modules" and "real-time analytics" that nobody supplied. A PASS therefore only means the email is well structured, not that it is truthful. The checker also uses simple substring matching, so it can be fooled.

**Ex 3 – With vs without search (Coursera):**
- **Without search:** 526 words, 0 source URLs, 4.1 s. It did say verification had not been performed. But it still listed facts from the model's general knowledge (founding year, headquarters, ticker) under "Verified Facts", which is contradictory because nothing was verified.
- **With search:** 418 words, 5 source URLs, 50.4 s. It separated sourced facts, inferred needs and unknowns.

Search clearly improves evidence quality, but it is slower and still imperfect. Some facts came from third-party sources (Wikipedia, a Substack analysis), and the figure labelled "FY2024 revenue $179.2M" appears to be the fourth-quarter figure, not the full year, so it must be checked against the company's investor page.

**Ex 4 – Approval gate:** The first attempt to send without approval was blocked with a `PermissionError`. After the human typed APPROVE, the stub send ran and the decision was logged with a timestamp and the structure-check result. The human step covers what the checker cannot. In the reviewed draft, the sentence "I've been following Northstar Learning's work in online education" is an unsupported claim, and the placeholders [Your Name] and [Your Company] were still present.

**Overall conclusion:** Tools make agents more useful, but each has limits. Rule-based tools miss context, search results need verification, and a human must review every draft before any external communication.